# Лабораторна робота №6 (Варіант 2)
## Інтелектуальні агенти LlamaIndex: визначення інструментів та керування пам'яттю

**Дисципліна:** Технології та системи нейронних мереж  
**Університет:** Національний університет «Львівська політехніка»  
**Тема:** Інтелектуальні агенти в LlamaIndex (Variant 2: Memory + Multi-Step Tool Orchestration)  

### Мета роботи
Дослідити принципи побудови та функціонування агентних систем на основі сучасного фреймворку `LlamaIndex` (workflow-first архітектура). Сконструювати набір спеціалізованих інструментів: роботу зі структурованим реєстром MLOps моделей, пошук у регламентній документації (incident runbook) та обчислювальний модуль відповідності SLO. Дослідити механізм контекстної пам'яті (`ChatMemoryBuffer`) для збереження стану між багаторазовими сесіями діалогу.

In [1]:
import os
import sys
import json
import asyncio
from pathlib import Path

# Незалежне визначення базових директорій
def resolve_lab_dir() -> Path:
    current = Path.cwd().resolve()
    if (current / 'data').exists() and (current / 'outputs').exists():
        return current
    if (current / 'lab6_llamaindex_agents').exists():
        return current / 'lab6_llamaindex_agents'
    for p in [current] + list(current.parents):
        cand = p / 'lab6_llamaindex_agents'
        if cand.exists():
            return cand
    return current

LAB_DIR = resolve_lab_dir()
DATA_DIR = LAB_DIR / 'data'
OUTPUTS_DIR = LAB_DIR / 'outputs'
print(f'Active Lab 6 Root Directory: {LAB_DIR}')
print(f'Data Directory: {DATA_DIR}')
print(f'Outputs Directory: {OUTPUTS_DIR}')

Active Lab 6 Root Directory: C:\Users\Eclipse\PycharmProjects\neural-network-technologies-and-systems\max\lab6_llamaindex_agents
Data Directory: C:\Users\Eclipse\PycharmProjects\neural-network-technologies-and-systems\max\lab6_llamaindex_agents\data
Outputs Directory: C:\Users\Eclipse\PycharmProjects\neural-network-technologies-and-systems\max\lab6_llamaindex_agents\outputs


### 1. Визначення та юніт-тестування інструментів агента

Для забезпечення прозорої діагностики промислової ML-інфраструктури агент оснащується чотирма детермінованими інструментами:
1. `query_model_registry`: пошук телеметрії конкретної моделі в JSON-реєстрі.
2. `list_degraded_models`: фільтрація всіх моделей зі статусом, відмінним від healthy.
3. `consult_incident_runbook`: повнотекстовий пошук за ключовими словами у регламенті ліквідації інцидентів.
4. `calculate_slo_margin`: математичний розрахунок відхилення реальних затримок/доступності від цільового SLO.

In [2]:
def query_model_registry(model_id: str) -> str:
    """Query the MLOps model registry for metadata and live operational telemetry."""
    reg_path = DATA_DIR / 'models_registry.json'
    with open(reg_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    for m in data['models']:
        if m['model_id'].lower() == model_id.strip().lower():
            return json.dumps(m, indent=2)
    valid = [m['model_id'] for m in data['models']]
    return f"Model '{model_id}' not found. Available models: {valid}"

def list_degraded_models() -> str:
    """List all production models that are currently in a degraded or failing state."""
    reg_path = DATA_DIR / 'models_registry.json'
    with open(reg_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    degraded = [m for m in data['models'] if m['status'] != 'healthy']
    return json.dumps(degraded, indent=2)

def consult_incident_runbook(query: str) -> str:
    """Search the MLOps incident runbook for triage protocols, mitigations, and rollback steps."""
    runbook_path = DATA_DIR / 'mlops_runbook.txt'
    with open(runbook_path, 'r', encoding='utf-8') as f:
        content = f.read()
    sections = content.split('## ')
    matched = []
    q_words = set(query.lower().replace('_', ' ').split())
    for sec in sections[1:]:
        sec_lower = sec.lower()
        if any(w in sec_lower for w in q_words):
            matched.append('## ' + sec.strip())
    if matched:
        return '\n\n'.join(matched[:2])
    return content[:1200]

def calculate_slo_margin(metric_value: float, slo_target: float, metric_type: str = 'latency') -> str:
    """Calculate compliance margin against SLO target. metric_type can be 'latency' or 'availability'."""
    if metric_type.lower() == 'latency':
        diff = metric_value - slo_target
        pct = (diff / slo_target) * 100.0
        status = 'VIOLATION' if diff > 0 else 'COMPLIANT'
        return json.dumps({
            'metric_type': 'latency',
            'value_ms': metric_value,
            'slo_target_ms': slo_target,
            'diff_ms': round(diff, 2),
            'margin_pct': round(pct, 2),
            'status': status,
            'severity': 'Critical' if pct > 20 else ('Warning' if pct > 0 else 'Nominal')
        })
    else:
        diff = metric_value - slo_target
        status = 'COMPLIANT' if diff >= 0 else 'VIOLATION'
        return json.dumps({
            'metric_type': 'availability',
            'value_pct': metric_value,
            'slo_target_pct': slo_target,
            'diff_pct': round(diff, 3),
            'status': status,
            'severity': 'Critical' if diff < -1.0 else ('Warning' if diff < 0 else 'Nominal')
        })

# Тестування роботи інструментів
print('=== Тест 1: query_model_registry(fraud_detector_v2) ===')
print(query_model_registry('fraud_detector_v2'))

print('\n=== Тест 2: list_degraded_models() ===')
print(list_degraded_models()[:250] + '...')

print('\n=== Тест 3: calculate_slo_margin(315.0, 250.0, latency) ===')
print(calculate_slo_margin(315.0, 250.0, 'latency'))

=== Тест 1: query_model_registry(fraud_detector_v2) ===
{
  "model_id": "fraud_detector_v2",
  "framework": "XGBoost",
  "version": "2.1.0",
  "status": "healthy",
  "p99_latency_ms": 64.0,
  "p50_latency_ms": 22.0,
  "error_rate": 0.0012,
  "availability_pct": 99.98,
  "slo_latency_ms": 100.0,
  "slo_availability_pct": 99.95,
  "requests_per_sec": 4500,
  "owner": "team-risk",
  "deployed_at": "2026-09-28T14:15:00Z"
}

=== Тест 2: list_degraded_models() ===
[
  {
    "model_id": "churn_predictor_v3",
    "framework": "LightGBM",
    "version": "3.4.1",
    "status": "degraded",
    "p99_latency_ms": 315.0,
    "p50_latency_ms": 110.0,
    "error_rate": 0.038,
    "availability_pct": 98.62,
    "slo_late...

=== Тест 3: calculate_slo_margin(315.0, 250.0, latency) ===
{"metric_type": "latency", "value_ms": 315.0, "slo_target_ms": 250.0, "diff_ms": 65.0, "margin_pct": 26.0, "status": "VIOLATION", "severity": "Critical"}


### 2. Ініціалізація LlamaIndex AgentWorkflow та локальної моделі

Використовуємо локальний OpenAI-сумісний ендпоінт Ollama (`qwen3:4b-instruct`) через адаптер `OpenAILike` із підтримкою Function Calling та сучасну декларативну структуру `AgentWorkflow` з пакету `llama_index.core.agent.workflow`.

In [3]:
from llama_index.llms.openai_like import OpenAILike
from llama_index.core.agent.workflow import AgentWorkflow
from llama_index.core.memory import ChatMemoryBuffer

llm = OpenAILike(
    model='qwen3:4b-instruct',
    api_base='http://127.0.0.1:11434/v1',
    api_key='ollama',
    is_chat_model=True,
    is_function_calling_model=True,
    context_window=4096,
    temperature=0.1
)

SYSTEM_PROMPT = (
    'You are an expert MLOps Incident Copilot agent at Lviv Polytechnic National University. '
    'You monitor machine learning systems, audit service-level objectives (SLO), and consult runbooks. '
    'Always check the registry for live metrics, calculate exact SLO margins using tools, and consult '
    'the runbook for concrete remediation steps. Refuse destructive operations outside your tool scope.'
)

tools = [query_model_registry, list_degraded_models, consult_incident_runbook, calculate_slo_margin]

agent = AgentWorkflow.from_tools_or_functions(
    tools,
    llm=llm,
    system_prompt=SYSTEM_PROMPT,
    verbose=False
)
print('LlamaIndex AgentWorkflow успішно ініціалізовано з 4 інструментами.')

LlamaIndex AgentWorkflow успішно ініціалізовано з 4 інструментами.


### 3. Дослідження та верифікація контекстної пам'яті (ChatMemoryBuffer)

Проведемо двоетапний діалог: у першому зверненні повідомляємо контекст проблеми без прямого запиту на розрахунок, а у другому зверненні перевіряємо, чи агент здатен використати інформацію з буфера пам'яті.

In [4]:
memory = ChatMemoryBuffer.from_defaults()

# Turn 1: Введення контексту дослідження
turn1_prompt = 'We are currently reviewing the recommender_ranker_v5 model.'
print(f'User [Turn 1]: {turn1_prompt}')
handler1 = agent.run(user_msg=turn1_prompt, memory=memory)
response1 = await handler1
print(f'Agent [Turn 1]:\n{response1}\n')

# Turn 2: Референсне звернення до збереженого контексту
turn2_prompt = 'What is its P99 latency and is it currently healthy according to the registry?'
print(f'User [Turn 2]: {turn2_prompt}')
handler2 = agent.run(user_msg=turn2_prompt, memory=memory)
response2 = await handler2
print(f'Agent [Turn 2]:\n{response2}\n')

print(f'Кількість повідомлень, зафіксованих у ChatMemoryBuffer: {len(memory.get())}')

User [Turn 1]: We are currently reviewing the recommender_ranker_v5 model.


Agent [Turn 1]:
The `recommender_ranker_v5` model is currently **healthy** with the following key metrics:

- **P99 Latency**: 118.0 ms (below the SLO target of 150.0 ms)
- **P50 Latency**: 45.0 ms
- **Error Rate**: 0.004% (very low)
- **Availability**: 99.91% (exceeds the SLO target of 99.9%)
- **Requests per second**: 8,900 (high traffic, but within capacity)

### SLO Compliance Check:
- **Latency Margin**:  
  `calculate_slo_margin(118.0, 150.0, "latency")` → **+32.0 ms margin** (exceeds SLO)
- **Availability Margin**:  
  `calculate_slo_margin(99.91, 99.9, "availability")` → **+0.01% margin** (exceeds SLO)

✅ Both SLOs are **met and exceeded**. No immediate action is required.

Would you like to proceed with a deeper analysis (e.g., error breakdown, traffic trends) or check for other models in degraded states?

User [Turn 2]: What is its P99 latency and is it currently healthy according to the registry?


Agent [Turn 2]:
The **P99 latency** of the `recommender_ranker_v5` model is **118.0 ms**.

According to the model registry, the model is **currently healthy**.  

This is confirmed by the `status` field in the registry metadata, which explicitly states `"status": "healthy"`.  

Additionally, the P99 latency (118.0 ms) is well below the SLO target of 150.0 ms, further validating its performance.

Кількість повідомлень, зафіксованих у ChatMemoryBuffer: 6


### 4. Висновки до блокноту 1

- Реалізовано та протестовано чотири детерміновані функції для взаємодії агента з MLOps інфраструктурою.
- Побудовано декларативний агент на базі `AgentWorkflow` LlamaIndex із системним промптом, що регламентує межі відповідальності.
- Підтверджено коректне функціонування `ChatMemoryBuffer`: агент утримує сутність попередньої розмови (`recommender_ranker_v5`) та адресно викликає інструменти перевірки телеметрії без повторного введення назви моделі.